# Stock Price Time Series + Sentiment Analysis

This notebook performs:
- historical stock analysis
- trend/volatility/drawdown metrics
- news headline sentiment scoring
- moving-average visualization

In [ ]:
# If needed, uncomment and run once:
# !pip install yfinance pandas numpy matplotlib scikit-learn vaderSentiment

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.linear_model import LinearRegression
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

plt.style.use("seaborn-v0_8")

ModuleNotFoundError: No module named 'numpy'

In [ ]:
ticker = "AAPL"  # change this
period = "1y"
interval = "1d"
max_headlines = 20

In [ ]:
df = yf.download(ticker, period=period, interval=interval, auto_adjust=True, progress=False)
if df.empty:
    raise ValueError(f"No price data found for ticker '{ticker}'")

df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()
df.head()

In [ ]:
close = df["Close"].copy()
returns = close.pct_change().dropna()

x = np.arange(len(close)).reshape(-1, 1)
y = close.values.reshape(-1, 1)
model = LinearRegression().fit(x, y)
trend_slope = float(model.coef_[0][0])

annualized_volatility = float(returns.std() * np.sqrt(252))

rolling_max = close.cummax()
drawdown = (close - rolling_max) / rolling_max
max_drawdown = float(drawdown.min())

print(f"Trend slope (price units/day): {trend_slope:.4f}")
print(f"Annualized volatility: {annualized_volatility:.2%}")
print(f"Max drawdown: {max_drawdown:.2%}")

In [ ]:
def _title_from_news_item(item):
    """Yahoo news: title may be top-level or under content.title."""
    if not isinstance(item, dict):
        return None
    t = item.get("title")
    if isinstance(t, str) and t.strip():
        return t.strip()
    content = item.get("content")
    if isinstance(content, dict):
        for key in ("title", "summary", "description"):
            val = content.get(key)
            if isinstance(val, str) and val.strip():
                return val.strip()
    return None


tk = yf.Ticker(ticker)
raw_news = getattr(tk, "news", []) or []
headlines = []
for n in raw_news[:max_headlines]:
    t = _title_from_news_item(n)
    if t:
        headlines.append(t)

analyzer = SentimentIntensityAnalyzer()
scores = [analyzer.polarity_scores(h)["compound"] for h in headlines]
sentiment_score = float(np.mean(scores)) if scores else 0.0

def sentiment_label(score: float, n_headlines: int) -> str:
    if n_headlines == 0:
        return "No news data"
    if score >= 0.1:
        return "Positive"
    if score <= -0.1:
        return "Negative"
    return "Neutral"

print(f"Sentiment score: {sentiment_score:.3f} ({sentiment_label(sentiment_score, len(headlines))})")
print(f"Headlines used: {len(headlines)}")
headlines[:5]

In [ ]:
ma20 = close.rolling(20).mean()
ma50 = close.rolling(50).mean()

plt.figure(figsize=(11, 6))
plt.plot(close.index, close.values, label="Close", linewidth=1.8)
plt.plot(ma20.index, ma20.values, label="MA20", linewidth=1.2)
plt.plot(ma50.index, ma50.values, label="MA50", linewidth=1.2)
plt.title(f"{ticker} Price Time Series")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.tight_layout()
plt.show()